In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# End-to-End E-Commerce Assistant: Orchestration & Evaluation with ADK

<a target="_blank" href="https://colab.research.google.com/github/google/adk-samples/blob/main/python/notebooks/orchestration/e2e_retail_assistant_adk.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

| Author(s) |
| --- |
| [Aniket Agrawal](https://github.com/aniketagrawal2012) |

## Overview

In this notebook, we build an **End-to-End E-Commerce Assistant** using the [Agent Development Kit (ADK)](https://google.github.io/adk-docs/). This agent goes beyond simple Q&A by intelligently orchestrating multiple Python-based tools to fulfill complex customer requests.

This notebook covers the complete agent lifecycle:

1.  **Tool Development**: Create custom Python functions for inventory search, stock checking, and discount calculation.
2.  **Agent Definition**: Package the tools into an ADK `Agent` powered by `gemini-2.5-flash`.
3.  **Local Testing**: Use `InMemoryRunner` to simulate customer interactions and verify dynamic tool calling.
4.  **LLM-as-a-Judge Evaluation**: Use ADK's `rubric_based_final_response_quality_v1` metric to automatically grade the agent's performance against strict business rules.

## 1. Setup & Installation

In [1]:
%pip install --upgrade --quiet "google-adk>=1.18.0" "google-cloud-aiplatform[evaluation]>=1.100.0" "google-adk[eval]"

import sys
if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()

    import IPython
    app = IPython.Application.instance()
    app.kernel.do_shutdown(True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.2/244.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 100.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.53.0 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.12.5 which is incompatible.
db-dtypes 1.6.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 which is incompatible.


## 2. Project Configuration

We need to configure our Google Cloud project. ADK uses `gemini-2.5-flash` on Vertex AI under the hood to power both the Agent and the Evaluation Judge.

In [10]:
import os

# Replace with your Google Cloud Project ID
PROJECT_ID = "aniket-personal"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

# Create an ADK App Workspace directory
APP_NAME = "retail_assistant"
APP_PATH = f"./{APP_NAME}"

!mkdir -p {APP_PATH}
# Properly initialize the package to expose the agent module
!echo 'from . import agent' > {APP_PATH}/__init__.py

print(f"Project {PROJECT_ID} configured. Agent workspace created at {APP_PATH}/")

Project aniket-personal configured. Agent workspace created at ./retail_assistant/


## 3. Application Architecture (Agent & Tools)

To properly evaluate an ADK agent using the ADK Evaluation CLI (`adk eval`), we must define our agent in an `agent.py` file within our App directory.

Below, we define three tools:
- `search_products`: Finds the Product ID and base price.
- `check_stock`: Verifies if an item is available.
- `apply_discount`: Calculates the adjusted price if the user has a promo code.

In [11]:
%%writefile {APP_PATH}/agent.py
from google.adk.agents import Agent

# Mock Database
INVENTORY = {
    "P001": {"name": "Wireless Noise-Canceling Headphones", "price": 299.99, "stock": 15, "category": "Electronics"},
    "P002": {"name": "Smart Watch Series 5", "price": 399.50, "stock": 0, "category": "Electronics"},
    "P003": {"name": "Ergonomic Office Chair", "price": 199.00, "stock": 5, "category": "Furniture"}
}

def search_products(query: str) -> list[dict]:
    """Search for products by name or category."""
    results = []
    for pid, details in INVENTORY.items():
        if query.lower() in details['name'].lower() or query.lower() in details['category'].lower():
            results.append({"product_id": pid, "name": details['name'], "price": details['price']})
    return results

def check_stock(product_id: str) -> str:
    """Check the available stock for a specific product ID."""
    product = INVENTORY.get(product_id)
    if not product:
        return f"Product {product_id} not found."
    if product["stock"] > 0:
        return f"In Stock: {product['stock']} available."
    return "Out of Stock."

def apply_discount(price: float, code: str) -> float:
    """Apply a discount code to a price and return the new price."""
    if code == "SAVE20":
        return round(price * 0.80, 2)
    elif code == "SAVE10":
        return round(price * 0.90, 2)
    return price

AGENT_INSTRUCTION = """
You are an expert E-Commerce Assistant. Help users find products, check stock, and calculate final prices.
Follow these rules strictly:
1. Always use `search_products` to find items based on user requests.
2. Once a product is identified, ALWAYS use `check_stock` to verify availability before recommending it.
3. If the user provides a discount code, use `apply_discount` on the product's base price.
4. Be polite, concise, and clearly state the final price and stock status.
"""

# Create the ADK Agent
root_agent = Agent(
    name="retail_assistant",
    model="gemini-2.5-flash",
    instruction=AGENT_INSTRUCTION,
    tools=[search_products, check_stock, apply_discount],
)


Overwriting ./retail_assistant/agent.py


## 4. Local Testing with InMemoryRunner

Before evaluating the agent programmatically, let's run a few interactive tests to see the tool choreography in action.

In [13]:
import sys
if "." not in sys.path:
    sys.path.append(".")

# Import the agent we just wrote to disk
from retail_assistant.agent import agent as retail_agent
from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(agent=retail_agent, app_name="retail_assistant")

print("\n=== Test 1: Out of Stock Item ===")
await runner.run_debug("I want to buy the Smart Watch Series 5. Is it available?")

print("\n=== Test 2: In Stock + Discount Code ===")
await runner.run_debug("I am looking for the Wireless Noise-Canceling Headphones. I have the code SAVE20. Check stock and tell me the final price.")


=== Test 1: Out of Stock Item ===
retail_assistant > I'm sorry, the Smart Watch Series 5 is currently out of stock.

=== Test 2: In Stock + Discount Code ===
retail_assistant > Great news! The Wireless Noise-Canceling Headphones are in stock (15 available). With your SAVE20 discount, the final price is $239.99.


[Event(model_version='gemini-2.5-flash', content=Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'query': 'Wireless Noise-Canceling Headphones'
         },
         id='adk-6493bb51-76d9-4fd6-8387-7e6efa909fc2',
         name='search_products'
       ),
       thought_signature=b'\n\xcc\x03\x01\x8f=k_AD\xff_\x87\xc1\xc7\xd6\x03\xf3S\xac\\\xbb\xd1\xfd\xc1\xa7n\xff\xc1\xfak\xf3Aj\xbf7\xb2\x11\x1b*V\xf2sk$\xfd<\xd1\xc6>\xf8\xef\xcb\xf90C\x87\xf0\x1c<8\x81\xbbehu\xfd\xdcY\xde\x1e\x7f\xb0\xe6\xd0\x96\xea\\&\x91J3\x97\xbf\xf6\xc6\x90\xa4\x06\xdb\x0f\x9b\xe9\xaam_!...'
     ),
   ],
   role='model'
 ), grounding_metadata=None, partial=None, turn_complete=None, finish_reason=<FinishReason.STOP: 'STOP'>, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=GenerateContentResponseUsageMetadata(
   candidates_token_count=10,
   candidates_tokens_details=[
     ModalityTokenCount(
       modality=<MediaModality.TEX

## 5. Setting Up the ADK Evaluation

Testing manually is great, but to scale our QA, we use ADK Evaluations. We'll define a Golden Dataset (an `evalset`) and evaluate it using an LLM-as-a-judge metric: `rubric_based_final_response_quality_v1`.

This checks if our agent adheres to our business rules (verifying stock and calculating discounts).

In [5]:
# Define the Evaluation Dataset (JSON format required by ADK)
serialized_eval_set = """
{
  "eval_set_id": "retail_eval_set_01",
  "name": "retail_eval_set_01",
  "eval_cases": [
    {
      "eval_id": "buy_headphones_discount",
      "conversation": [
        {
          "invocation_id": "inv-01",
          "user_content": {
            "parts": [{"text": "I want to buy the Wireless Noise-Canceling Headphones. I have the code SAVE20. Check stock and price."}]
          },
          "final_response": {
            "parts": [{"text": "Great choice! The Wireless Noise-Canceling Headphones are in stock (15 available). The base price is $299.99, but with your SAVE20 code, the final discounted price is $239.99."}]
          }
        }
      ],
      "session_input": {
        "app_name": "retail_assistant",
        "user_id": "user"
      }
    }
  ]
}
"""

# Write the evalset file to the app directory
!echo '{serialized_eval_set}' > {APP_PATH}/retail_eval_set_01.evalset.json
print("Evaluation dataset saved to retail_eval_set_01.evalset.json")

Evaluation dataset saved to retail_eval_set_01.evalset.json


### Configure the LLM Judge Rubric

We instruct the Judge Model (`gemini-2.5-flash`) on exactly what to grade. Here we ensure the agent mentioned stock status and applied the final discount correctly.

In [8]:
eval_config = """
{
  "criteria": {
    "rubric_based_final_response_quality_v1": {
      "threshold": 0.8,
      "judge_model_options": {
        "judge_model": "gemini-2.5-flash",
        "num_samples": 3
      },
      "rubrics": [
        {
          "rubric_id": "stock_check",
          "rubric_content": {
            "text_property": "The response clearly states whether the item is in stock or out of stock."
          }
        },
        {
          "rubric_id": "discount_applied",
          "rubric_content": {
            "text_property": "The response calculates and provides the final discounted price."
          }
        }
      ]
    }
  }
}
"""

!echo '{eval_config}' > {APP_PATH}/eval_config.json
print("Evaluation config saved to eval_config.json")

Evaluation config saved to eval_config.json


## 6. Run the ADK Evaluation

Using the ADK CLI, we trigger the evaluation process. This reads the config, spins up the LLM judge, evaluates our test case, and reports a Pass/Fail status alongside the specific rubric reasoning.

In [12]:
!adk eval \
    {APP_PATH} \
    --config_file_path {APP_PATH}/eval_config.json \
    retail_eval_set_01 \
    --print_detailed_results \
    --log_level=CRITICAL

/usr/local/lib/python3.12/dist-packages/google/adk/evaluation/metric_evaluator_registry.py:112: UserWarning: [EXPERIMENTAL] MetricEvaluatorRegistry: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  metric_evaluator_registry = MetricEvaluatorRegistry()
/usr/local/lib/python3.12/dist-packages/google/adk/evaluation/local_eval_service.py:124: UserWarning: [EXPERIMENTAL] UserSimulatorProvider: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  user_simulator_provider: UserSimulatorProvider = UserSimulatorProvider(),
Using evaluation criteria: criteria={'rubric_based_final_response_quality_v1': BaseCriterion(threshold=0.8, include_intermediate_responses_in_final=False, judge_model_options={'judge_model': 'gemini-2.5-flash', 'num_samples': 3}, rubrics=[{'rubric_id': 'stock_check', 'rubric_content': {'text_proper

## Summary

Congratulations! You've successfully built and tested an E-Commerce Assistant capable of complex tool orchestration.

By adopting ADK's programmatic approach and evaluation pipelines, you guarantee that your agent consistently adheres to crucial business processes—such as checking inventory bounds and honoring discount math—before it ever touches a production environment.